# Registro histórico: 13_checkpoint_comparison

Origem: `C51` — `Colab Notebooks/Summary.ipynb`.

Avaliação de checkpoints com AUC, sensibilidade, especificidade e limiar.

**Limitações conhecidas:** Resumo narrativo contradiz saída MobileNet; células dependem de variáveis anteriores e checkpoints ausentes.

Cópia organizada em 25/10/2026. Código e texto originais preservados abaixo; saídas, contagens de execução e metadados do Colab removidos. As métricas salvas estão em `results/historical_evidence.json`. Não executar em lote: estes cadernos dependem do ambiente e de arquivos originais. Veja `docs/experiments.md` para o fluxo novo.


In [ ]:
#@title 1) Montar Google Drive e definir diretórios
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
!cp -r "/content/drive/MyDrive/Trained_Models" "/content/"

In [ ]:


# Ajuste se você usa outro caminho
LOCAL_MODELS_DIR = Path("/content/Trained_Models")
LOCAL_MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Existem arquivos?", list(LOCAL_MODELS_DIR.glob("*.pth"))[:5])


In [ ]:
#@title 2) Funções utilitárias: criar modelo e carregar pesos
import torch
import torch.nn as nn
import torchvision.models as models
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def create_model(arch: str, num_classes: int = 2):
    arch = arch.lower().strip()
    if arch in ["vgg16", "vgg-16"]:
        model = models.vgg16(weights=None)
        in_features = model.classifier[-1].in_features  # último Linear
        model.classifier[-1] = nn.Linear(in_features, num_classes)
        model_name = "VGG16"
    elif arch in ["mobilenetv3-large", "mobilenet_v3_large", "mobilenetv3"]:
        model = models.mobilenet_v3_large(weights=None)
        in_features = model.classifier[-1].in_features  # último Linear
        model.classifier[-1] = nn.Linear(in_features, num_classes)
        model_name = "MobileNetV3-Large"
    else:
        raise ValueError(f"Arquitetura não reconhecida: {arch}")
    return model.to(device), model_name

def load_state_dict_safely(model, ckpt_path):
    state = torch.load(ckpt_path, map_location=device)
    # Alguns treinos salvam em "state_dict"
    if isinstance(state, dict) and "state_dict" in state and isinstance(state["state_dict"], dict):
        state = state["state_dict"]

    cleaned = {}
    for k, v in state.items():
        if k.startswith("_orig_mod."):
            k = k[len("_orig_mod."):]
        if k.startswith("module."):   # caso tenha sido treinado com DataParallel/DDP
            k = k[len("module."):]
        cleaned[k] = v

    missing, unexpected = model.load_state_dict(cleaned, strict=False)  # strict=True se quiser travar
    if missing:
        print("⚠️ Missing keys:", missing[:8], "..." if len(missing) > 8 else "")
    if unexpected:
        print("⚠️ Unexpected keys:", unexpected[:8], "..." if len(unexpected) > 8 else "")
    return model

def maybe_load_classes_from_history(history_path):
    try:
        with open(history_path, "r") as f:
            h = json.load(f)
        if "classes" in h and isinstance(h["classes"], (list, tuple)) and len(h["classes"])>0:
            return list(h["classes"])
    except Exception:
        pass
    return ["ben", "mel"]  # fallback comum do seu projeto


In [ ]:
#@title 3) Escolher o checkpoint (.pth) no Drive
from datetime import datetime
from pathlib import Path

# Opção A) informe explicitamente o nome do arquivo .pth:
ckpt_name = "EfficientNetB0_SkinLesion_Binary_Classifier_BEST_F1.pth"  #@param {type:"string"}
# Ex.: "VGG16_SkinLesion_Binary_Classifier.pth" ou "MobileNetV3_SkinLesion_Binary_Classifier_1.pth"

# Opção B) se deixar vazio, tento buscar o MAIS RECENTE automaticamente:
def find_latest_pth(folder: Path):
    files = list(folder.glob("*.pth"))
    if not files:
        return None
    files.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return files[0]

ckpt_path = LOCAL_MODELS_DIR / ckpt_name if ckpt_name else find_latest_pth(LOCAL_MODELS_DIR)
assert ckpt_path and ckpt_path.exists(), "Não encontrei arquivo .pth no caminho indicado."

print("Checkpoint selecionado:", ckpt_path.name,
      f"({datetime.fromtimestamp(ckpt_path.stat().st_mtime)})")

# (opcional) copiar para pasta local por I/O mais rápido
local_ckpt_path = LOCAL_MODELS_DIR / ckpt_path.name
if not local_ckpt_path.exists():
    local_ckpt_path.write_bytes(ckpt_path.read_bytes())
print("Checkpoint local:", local_ckpt_path)


In [ ]:
#@title 4) Detectar arquitetura, reconstruir e carregar pesos
# Detecta pelo nome do arquivo, mas você pode forçar em 'force_arch'
force_arch = "efficientnet_b0"  #@param ["", "vgg16", "mobilenetv3-large", "efficientnet_b0"] {allow-input: true}
num_classes = 2  #@param {type:"integer"}

name_lower = local_ckpt_path.name.lower()
if force_arch:
    arch = force_arch
elif "vgg16" in name_lower or "vgg" in name_lower:
    arch = "vgg16"
elif "mobilenetv3" in name_lower:
    arch = "mobilenetv3-large"
elif "efficientnet" in name_lower:
    arch = "efficientnet_b0"
else:
    # fallback seguro: peça para informar
    arch = "efficientnet_b0"  # <- troque se necessário

model, model_name = create_model(arch, num_classes=num_classes)
model = load_state_dict_safely(model, local_ckpt_path)
model.eval()

print(f"✅ Modelo carregado: {model_name} | num_classes={num_classes}")


In [ ]:
#@title 5) (Opcional) Carregar classes do histórico JSON pareado
# Tenta achar um .json "vizinho" com o mesmo prefixo do modelo
hist_json = None
cand_jsons = list(LOCAL_MODELS_DIR.glob("*.json"))
if cand_jsons:
    # heurística: procura json com mesmo prefixo do pth
    stem = local_ckpt_path.stem.split("_")[0]  # p.ex. "VGG16" ou "MobileNetV3"
    matches = [j for j in cand_jsons if stem.lower() in j.name.lower()]
    hist_json = matches[1] if matches else cand_jsons[0]

classes = maybe_load_classes_from_history(hist_json) if hist_json else ["ben", "mel"]
print("Classes:", classes, "| history:", hist_json.name if hist_json else "não encontrado")


In [ ]:
#@title 6) Inferência rápida em uma imagem (opcional)
from PIL import Image
from torchvision import transforms
from torchvision.models import VGG16_Weights, MobileNet_V3_Large_Weights
import torch.nn.functional as F
import numpy as np

# Normalização padrão por arquitetura
if "vgg" in model_name.lower():
    mean = VGG16_Weights.IMAGENET1K_V1.transforms().mean
    print("vgg")
    std  = VGG16_Weights.IMAGENET1K_V1.transforms().std
else:
    mean = MobileNet_V3_Large_Weights.IMAGENET1K_V1.transforms().mean
    std  = MobileNet_V3_Large_Weights.IMAGENET1K_V1.transforms().std

eval_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

img_path = "/content/drive/MyDrive/0-vgg/base_dir/test_dir/mel/ISIC_0024315.jpg"  #@param {type:"string"}
if img_path:
    img = Image.open(img_path).convert("RGB")
    x = eval_transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x)
        probs = F.softmax(logits, dim=1).cpu().numpy()[0]
    for cls, p in zip(classes, probs):
        print(f"{cls}: {p*100:.2f}%")
    print("Predição:", classes[int(np.argmax(probs))])
else:
    print("Informe 'img_path' (caminho da imagem) para testar a inferência.")


In [ ]:
# -*- coding: utf-8 -*-
"""Summary_EfficientNetB0.ipynb

Versão apenas para EfficientNet-B0
"""

#@title 1) Montar Google Drive e definir diretórios
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
!cp -r "/content/drive/MyDrive/Trained_Models" "/content/"

# Pasta local onde os .pth e .json estão
LOCAL_MODELS_DIR = Path("/content/Trained_Models")
LOCAL_MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Existem arquivos .pth?", list(LOCAL_MODELS_DIR.glob("*.pth"))[:5])

#@title 2) Funções utilitárias: criar modelo EfficientNet-B0 e carregar pesos
import torch
import torch.nn as nn
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def create_model_efficientnet_b0(num_classes: int = 2):
    """
    Cria EfficientNet-B0 para classificação binária.
    """
    model = models.efficientnet_b0(weights=None)
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)
    model_name = "EfficientNet-B0"
    return model.to(device), model_name

def load_state_dict_safely(model, ckpt_path):
    """
    Carrega state_dict tratando casos com 'state_dict', 'module.' e '_orig_mod.'.
    """
    state = torch.load(ckpt_path, map_location=device)
    if isinstance(state, dict) and "state_dict" in state and isinstance(state["state_dict"], dict):
        state = state["state_dict"]

    cleaned = {}
    for k, v in state.items():
        if k.startswith("_orig_mod."):
            k = k[len("_orig_mod."):]
        if k.startswith("module."):
            k = k[len("module."):]
        cleaned[k] = v

    missing, unexpected = model.load_state_dict(cleaned, strict=False)
    if missing:
        print("⚠️ Missing keys:", missing[:8], "..." if len(missing) > 8 else "")
    if unexpected:
        print("⚠️ Unexpected keys:", unexpected[:8], "..." if len(unexpected) > 8 else "")
    return model

def maybe_load_classes_from_history(history_path):
    """
    Tenta ler as classes de um JSON de histórico.
    """
    try:
        with open(history_path, "r") as f:
            h = json.load(f)
        if "classes" in h and isinstance(h["classes"], (list, tuple)) and len(h["classes"]) > 0:
            return list(h["classes"])
    except Exception:
        pass
    # fallback binário padrão
    return ["ben", "mel"]

#@title 3) Escolher o checkpoint (.pth) do EfficientNet-B0
from datetime import datetime

# Opção A) informe explicitamente o nome do arquivo .pth (caso já saiba):
ckpt_name = "EfficientNetB0_SkinLesion_Binary_Classifier_BEST_F1.pth"  #@param {type:"string"}

# Opção B) se ckpt_name == "", pegar o mais recente em LOCAL_MODELS_DIR
def find_latest_pth(folder: Path):
    files = list(folder.glob("*.pth"))
    if not files:
        return None
    files.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return files[0]

ckpt_path = LOCAL_MODELS_DIR / ckpt_name if ckpt_name else find_latest_pth(LOCAL_MODELS_DIR)
assert ckpt_path and ckpt_path.exists(), "Não encontrei arquivo .pth no caminho indicado."

print("Checkpoint selecionado:", ckpt_path.name,
      f"({datetime.fromtimestamp(ckpt_path.stat().st_mtime)})")

# Copiar para pasta local (se quiser evitar I/O no Drive; aqui já está em /content, mas mantemos a lógica)
local_ckpt_path = LOCAL_MODELS_DIR / ckpt_path.name
if not local_ckpt_path.exists():
    local_ckpt_path.write_bytes(ckpt_path.read_bytes())
print("Checkpoint local:", local_ckpt_path)

#@title 4) Construir EfficientNet-B0 e carregar pesos
num_classes = 2  #@param {type:"integer"}

model, model_name = create_model_efficientnet_b0(num_classes=num_classes)
model = load_state_dict_safely(model, local_ckpt_path)
model.eval()

print(f"✅ Modelo carregado: {model_name} | num_classes={num_classes}")

#@title 5) (Opcional) Carregar classes do histórico JSON pareado
from pathlib import Path

hist_json = None
cand_jsons = list(LOCAL_MODELS_DIR.glob("*.json"))
if cand_jsons:
    stem = local_ckpt_path.stem.split("_")[0]  # prefixo, ex: "EfficientNetB0"
    matches = [j for j in cand_jsons if stem.lower() in j.name.lower()]
    hist_json = matches[0] if matches else cand_jsons[0]

classes = maybe_load_classes_from_history(hist_json) if hist_json else ["ben", "mel"]
print("Classes:", classes, "| history:", hist_json.name if hist_json else "não encontrado")

#@title 6) Inferência rápida em uma imagem (opcional)
from PIL import Image
from torchvision import transforms
from torchvision.models import EfficientNet_B0_Weights
import torch.nn.functional as F
import numpy as np

# Normalização padrão do EfficientNet-B0 (ImageNet)
eff_weights = EfficientNet_B0_Weights.IMAGENET1K_V1
mean = eff_weights.transforms().mean
std  = eff_weights.transforms().std

eval_transform_single = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

img_path = ""  #@param {type:"string"}
# Exemplo: "/content/drive/MyDrive/0-efficientnet/base_dir/test_dir/mel/ISIC_XXXX.jpg"

if img_path:
    img = Image.open(img_path).convert("RGB")
    x = eval_transform_single(img).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x)
        probs = F.softmax(logits, dim=1).cpu().numpy()[0]
    for cls, p in zip(classes, probs):
        print(f"{cls}: {p*100:.2f}%")
    print("Predição:", classes[int(np.argmax(probs))])
else:
    print("Informe 'img_path' (caminho da imagem) para testar a inferência, se quiser.")

#@title 7) Datasets e DataLoaders (VAL/TEST)
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

DATA_ROOT = Path("/content")  # ajuste se necessário
val_dir  = DATA_ROOT / "val_dir"
test_dir = DATA_ROOT / "test_dir"

# Normalização EfficientNet-B0
mean = eff_weights.transforms().mean
std  = eff_weights.transforms().std

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

val_data  = datasets.ImageFolder(root=val_dir,  transform=eval_transform)
test_data = datasets.ImageFolder(root=test_dir, transform=eval_transform)

BATCH_SIZE = 32
val_loader  = DataLoader(val_data,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
                         pin_memory=(device.type == "cuda"))
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
                         pin_memory=(device.type == "cuda"))

class_names = val_data.classes
print("Classes (dataset):", class_names)

#@title 8) Métricas médicas (ROC/PR, sens/esp, PPV/NPV, MCC, Brier) @ limiar ótimo
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                             precision_recall_curve, confusion_matrix,
                             classification_report,
                             matthews_corrcoef, cohen_kappa_score, brier_score_loss)
from tqdm.auto import tqdm

# classe positiva: "mel" se existir; senão usa o índice 1
pos_label_name = "mel" if "mel" in class_names else class_names[min(1, len(class_names)-1)]
pos_label = class_names.index(pos_label_name)

@torch.no_grad()
def probs_and_labels(model, loader, pos_idx):
    model.eval()
    all_probs, all_y = [], []
    for X, y in tqdm(loader, leave=False, desc="Inferindo"):
        X = X.to(device, non_blocking=True)
        logits = model(X)
        p = F.softmax(logits, dim=1)[:, pos_idx].detach().cpu().numpy()
        all_probs.append(p)
        all_y.append(y.numpy())
    return np.concatenate(all_probs), np.concatenate(all_y)

# Val e Test
p_val,  y_val  = probs_and_labels(model, val_loader,  pos_label)
p_test, y_test = probs_and_labels(model, test_loader, pos_label)

y_val_bin  = (y_val  == pos_label).astype(int)
y_test_bin = (y_test == pos_label).astype(int)

# Limiar ótimo por Youden (na validação)
fpr, tpr, thr = roc_curve(y_val_bin, p_val)
best_idx = int(np.argmax(tpr - fpr))
t_star = float(thr[best_idx])

# Predições no TEST com o limiar escolhido na VAL
yhat_test = (p_test >= t_star).astype(int)

# Métricas principais no TEST
tn, fp, fn, tp = confusion_matrix(y_test_bin, yhat_test).ravel()
sens = tp / (tp + fn + 1e-12)  # recall da classe positiva (mel)
spec = tn / (tn + fp + 1e-12)
ppv  = tp / (tp + fp + 1e-12)
npv  = tn / (tn + fn + 1e-12)
acc  = (yhat_test == y_test_bin).mean()
bal_acc = 0.5*(sens + spec)
mcc  = matthews_corrcoef(y_test_bin, yhat_test)
kappa = cohen_kappa_score(y_test_bin, yhat_test)

roc_auc = roc_auc_score(y_test_bin, p_test)
pr_auc  = average_precision_score(y_test_bin, p_test)
brier   = brier_score_loss(y_test_bin, p_test)

print("="*60)
print("📊 EXTENDED TEST SUMMARY - EfficientNet-B0")
print("="*60)
print(f"Positive class: '{pos_label_name}' (idx {pos_label})")
print(f"Threshold (Youden@VAL): {t_star:.3f}")
print(f"Accuracy: {acc*100:.2f}% | Balanced Acc: {bal_acc*100:.2f}%")
print(f"ROC-AUC: {roc_auc:.3f} | PR-AUC ({pos_label_name}): {pr_auc:.3f}")
print(f"Sensitivity ({pos_label_name}): {sens*100:.2f}% | Specificity: {spec*100:.2f}%")
print(f"PPV: {ppv*100:.2f}% | NPV: {npv*100:.2f}%")
print(f"MCC: {mcc:.3f} | Cohen's κ: {kappa:.3f} | Brier: {brier:.4f}")
print("Confusion Matrix @thr* :")
print(f"TN={tn}  FP={fp}\nFN={fn}  TP={tp}")
print("="*60)

print("\n📈 Classification report (TEST @thr*):")
print(classification_report(
    y_test_bin,
    yhat_test,
    target_names=[f"non-{pos_label_name}", pos_label_name],
    digits=4
))

#@title 9) Curvas ROC/PR e calibração (Reliability)
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

# ROC
fpr_t, tpr_t, _ = roc_curve(y_test_bin, p_test)
plt.figure(figsize=(5, 4))
plt.plot(fpr_t, tpr_t, label=f"ROC (AUC={roc_auc:.3f})")
plt.plot([0, 1], [0, 1], '--', lw=1)
plt.xlabel("1 - Specificity (FPR)")
plt.ylabel("Sensitivity (TPR)")
plt.title("ROC Curve (TEST) - EfficientNet-B0")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# PR
prec, rec, _ = precision_recall_curve(y_test_bin, p_test)
plt.figure(figsize=(5, 4))
plt.plot(rec, prec, label=f"PR (AP={pr_auc:.3f})")
plt.xlabel(f"Recall ({pos_label_name})")
plt.ylabel(f"Precision ({pos_label_name})")
plt.title("Precision-Recall (TEST) - EfficientNet-B0")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Reliability (calibração)
frac_pos, mean_pred = calibration_curve(y_test_bin, p_test, n_bins=15, strategy="uniform")
plt.figure(figsize=(5, 4))
plt.plot([0, 1], [0, 1], '--', lw=1)
plt.plot(mean_pred, frac_pos, marker='o')
plt.xlabel("Confiança média (bin)")
plt.ylabel("Fração positiva observada")
plt.title("Reliability Diagram (TEST) - EfficientNet-B0")
plt.grid(alpha=0.3)
plt.show()

#@title 10) TRAINING SUMMARY enriquecido (checkpoint carregado)
import os

best_val_acc = None
epochs_run = None
total_train_time = None

if hist_json and os.path.exists(hist_json):
    try:
        with open(hist_json, "r") as f:
            H = json.load(f)
        best_val_acc = H.get("best_val_acc", None)
        epochs_run = H.get("epochs_run", None)
        total_train_time = H.get("total_train_time_sec", None)
    except Exception as e:
        print("Erro ao ler hist_json:", e)

print("="*60)
print("📊 TRAINING SUMMARY (Loaded EfficientNet-B0 Checkpoint)")
print("="*60)
print(f"Model: {model_name}")
print(f"Task: Binary Classification (Melanoma vs Non-Melanoma)")
print(f"Dataset Size: Val={len(val_data)}, Test={len(test_data)}")

if epochs_run is not None:
    print(f"Epochs Trained: {epochs_run}")
if best_val_acc is not None:
    print(f"Best Validation Accuracy (from history): {best_val_acc:.2f}%")
else:
    # Se não houver histórico, usamos a acurácia na VAL com o mesmo limiar:
    yhat_val = (p_val >= t_star).astype(int)
    val_acc = (yhat_val == y_val_bin).mean() * 100
    print(f"Validation Accuracy @thr*: {val_acc:.2f}%")

# Test metrics principais
print(f"Final Test Accuracy: {acc*100:.2f}%")
print(f"ROC-AUC (Test): {roc_auc:.3f} | PR-AUC ({pos_label_name}): {pr_auc:.3f}")
print(f"Sensitivity ({pos_label_name}): {sens*100:.2f}% | Specificity: {spec*100:.2f}%")
print(f"PPV: {ppv*100:.2f}% | NPV: {npv*100:.2f}% | Balanced Acc: {bal_acc*100:.2f}%")
print(f"MCC: {mcc:.3f} | Brier: {brier:.4f} | Threshold*: {t_star:.3f}")
print(f"Model Loaded From: {local_ckpt_path}")
print("="*60)

#@title 11) Plotar Matriz de Confusão
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_bin, yhat_test)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=[f"non-{pos_label_name}", pos_label_name],
            yticklabels=[f"non-{pos_label_name}", pos_label_name])
plt.xlabel("Predição")
plt.ylabel("Real")
plt.title("Matriz de Confusão (TEST) - EfficientNet-B0")
plt.show()

#@title 12) (Opcional) Curvas de treino/val (se existirem no JSON)
if hist_json and os.path.exists(hist_json):
    try:
        with open(hist_json, "r") as f:
            H = json.load(f)

        has_loss = "train_losses" in H and "val_losses" in H
        has_acc  = "train_accs" in H and "val_accs" in H

        if has_loss:
            plt.figure(figsize=(6, 5))
            plt.plot(H["train_losses"], label="Train Loss")
            plt.plot(H["val_losses"], label="Val Loss")
            plt.xlabel("Epoch")
            plt.ylabel("Loss")
            plt.title("Training and Validation Loss - EfficientNet-B0")
            plt.legend()
            plt.grid(alpha=0.3)
            plt.show()

        if has_acc:
            plt.figure(figsize=(6, 5))
            plt.plot(H["train_accs"], label="Train Accuracy")
            plt.plot(H["val_accs"], label="Val Accuracy")
            plt.xlabel("Epoch")
            plt.ylabel("Accuracy (%)")
            plt.title("Training and Validation Accuracy - EfficientNet-B0")
            plt.legend()
            plt.grid(alpha=0.3)
            plt.show()

        if not (has_loss or has_acc):
            print("JSON não contém 'train_losses/val_losses' ou 'train_accs/val_accs' para plotar curvas.")
    except Exception as e:
        print("Erro ao plotar curvas de treino/val:", e)
else:
    print("Nenhum arquivo de histórico encontrado para plotar curvas de treino/val.")


In [ ]:
#@title 7) Datasets e DataLoaders (VAL/TEST)
from pathlib import Path
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchvision.models import VGG16_Weights, MobileNet_V3_Large_Weights

DATA_ROOT = Path("/content")  # ajuste se necessário
val_dir  = DATA_ROOT / "val_dir"
test_dir = DATA_ROOT / "test_dir"

# Normalização conforme a arquitetura carregada
if "vgg" in model_name.lower():
    mean = VGG16_Weights.IMAGENET1K_V1.transforms().mean
    std  = VGG16_Weights.IMAGENET1K_V1.transforms().std
else:
    mean = MobileNet_V3_Large_Weights.IMAGENET1K_V1.transforms().mean
    std  = MobileNet_V3_Large_Weights.IMAGENET1K_V1.transforms().std

eval_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

val_data  = datasets.ImageFolder(root=val_dir,  transform=eval_transform)
test_data = datasets.ImageFolder(root=test_dir, transform=eval_transform)

BATCH_SIZE = 32
val_loader  = DataLoader(val_data,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
                         pin_memory=(device.type=="cuda"))
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
                         pin_memory=(device.type=="cuda"))

class_names = val_data.classes
print("Classes (dataset):", class_names)


In [ ]:
#@title 8) Métricas médicas (ROC/PR, sens/esp, PPV/NPV, MCC, Brier) @ limiar ótimo
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                             precision_recall_curve, confusion_matrix,
                             classification_report,
                             matthews_corrcoef, cohen_kappa_score, brier_score_loss)
from tqdm.auto import tqdm

# classe positiva: "mel" se existir; senão usa o índice 1
pos_label_name = "mel" if "mel" in class_names else class_names[min(1, len(class_names)-1)]
pos_label = class_names.index(pos_label_name)

@torch.no_grad()
def probs_and_labels(model, loader, pos_idx):
    model.eval()
    all_probs, all_y = [], []
    for X, y in tqdm(loader, leave=False, desc="Inferindo"):
        X = X.to(device, non_blocking=True)
        logits = model(X)
        p = F.softmax(logits, dim=1)[:, pos_idx].detach().cpu().numpy()
        all_probs.append(p); all_y.append(y.numpy())
    return np.concatenate(all_probs), np.concatenate(all_y)

# Val e Test
p_val,  y_val  = probs_and_labels(model, val_loader,  pos_label)
p_test, y_test = probs_and_labels(model, test_loader, pos_label)

y_val_bin  = (y_val  == pos_label).astype(int)
y_test_bin = (y_test == pos_label).astype(int)

# Limiar ótimo por Youden (na validação)
fpr, tpr, thr = roc_curve(y_val_bin, p_val)
best_idx = int(np.argmax(tpr - fpr))
t_star = float(thr[best_idx])

# Predições no TEST com o limiar escolhido na VAL
yhat_test = (p_test >= t_star).astype(int)

# Métricas principais no TEST
tn, fp, fn, tp = confusion_matrix(y_test_bin, yhat_test).ravel()
sens = tp / (tp + fn + 1e-12)  # recall da classe positiva (mel)
spec = tn / (tn + fp + 1e-12)
ppv  = tp / (tp + fp + 1e-12)
npv  = tn / (tn + fn + 1e-12)
acc  = (yhat_test == y_test_bin).mean()
bal_acc = 0.5*(sens + spec)
mcc  = matthews_corrcoef(y_test_bin, yhat_test)
kappa = cohen_kappa_score(y_test_bin, yhat_test)

roc_auc = roc_auc_score(y_test_bin, p_test)
pr_auc  = average_precision_score(y_test_bin, p_test)
brier   = brier_score_loss(y_test_bin, p_test)

print("="*60)
print("📊 EXTENDED TEST SUMMARY")
print("="*60)
print(f"Positive class: '{pos_label_name}' (idx {pos_label})")
print(f"Threshold (Youden@VAL): {t_star:.3f}")
print(f"Accuracy: {acc*100:.2f}% | Balanced Acc: {bal_acc*100:.2f}%")
print(f"ROC-AUC: {roc_auc:.3f} | PR-AUC ({pos_label_name}): {pr_auc:.3f}")
print(f"Sensitivity ({pos_label_name}): {sens*100:.2f}% | Specificity: {spec*100:.2f}%")
print(f"PPV: {ppv*100:.2f}% | NPV: {npv*100:.2f}%")
print(f"MCC: {mcc:.3f} | Cohen's κ: {kappa:.3f} | Brier: {brier:.4f}")
print("Confusion Matrix @thr* :")
print(f"TN={tn}  FP={fp}\nFN={fn}  TP={tp}")
print("="*60)

# Classification report por classe (usando o limiar escolhido)
# Para isso, precisamos transformar yhat_test (binário) em rótulos 0..C-1
# Aqui construímos um vetor de rótulos onde a classe pos_label recebe 1 e a outra 0:
y_pred_labels = (yhat_test == 1).astype(int)  # binário 0/1 com "mel"=1 se ela for pos_label
# Se necessário, remapeie para 0..C-1 na mesma ordem de class_names (para 2 classes funciona)
print("\n📈 Classification report (TEST @thr*):")
print(classification_report(y_test_bin, y_pred_labels,
                            target_names=[c for c in ["non-mel","mel"]][:2],
                            digits=4))


In [ ]:
#@title 9) Curvas ROC/PR e calibração (Reliability)
import matplotlib.pyplot as plt
import numpy as np
from sklearn.calibration import calibration_curve

# ROC
fpr_t, tpr_t, _ = roc_curve(y_test_bin, p_test)
plt.figure(figsize=(5,4))
plt.plot(fpr_t, tpr_t, label=f"ROC (AUC={roc_auc:.3f})")
plt.plot([0,1],[0,1],'--',lw=1)
plt.xlabel("1 - Specificity (FPR)"); plt.ylabel("Sensitivity (TPR)")
plt.title("ROC Curve (TEST)"); plt.legend(); plt.grid(alpha=0.3); plt.show()

# PR
prec, rec, _ = precision_recall_curve(y_test_bin, p_test)
plt.figure(figsize=(5,4))
plt.plot(rec, prec, label=f"PR (AP={pr_auc:.3f})")
plt.xlabel("Recall (mel)"); plt.ylabel("Precision (mel)")
plt.title("Precision-Recall (TEST)"); plt.legend(); plt.grid(alpha=0.3); plt.show()

# Reliability (calibração)
frac_pos, mean_pred = calibration_curve(y_test_bin, p_test, n_bins=15, strategy="uniform")
plt.figure(figsize=(5,4))
plt.plot([0,1],[0,1],'--',lw=1)
plt.plot(mean_pred, frac_pos, marker='o')
plt.xlabel("Confiança média (bin)"); plt.ylabel("Fração positiva observada")
plt.title("Reliability Diagram (TEST)"); plt.grid(alpha=0.3); plt.show()


In [ ]:
#@title 10) TRAINING SUMMARY enriquecido (checkpoint carregado)
import json, os

# Tenta extrair dados do histórico (se a célula de import já achou 'hist_json')
best_val_acc = None
epochs_run = None
total_train_time = None
if 'hist_json' in globals() and hist_json and os.path.exists(hist_json):
    try:
        with open(hist_json, "r") as f:
            H = json.load(f)
        best_val_acc = H.get("best_val_acc", None)
        epochs_run = H.get("epochs_run", None)
        total_train_time = H.get("total_train_time_sec", None)
    except Exception:
        pass

print("="*60)
print("📊 TRAINING SUMMARY (Loaded Checkpoint)")
print("="*60)
print(f"Model: {model_name}")
print(f"Task: Binary Classification (Melanoma vs Non-Melanoma)")
print(f"Dataset Size: Val={len(val_data)}, Test={len(test_data)}")
if epochs_run is not None:
    print(f"Epochs Trained: {epochs_run}")
if best_val_acc is not None:
    print(f"Best Validation Accuracy (from history): {best_val_acc:.2f}%")
else:
    # Se não houver histórico, mostramos a acurácia de validação com o mesmo limiar:
    yhat_val = (p_val >= t_star).astype(int)
    val_acc = (yhat_val == y_val_bin).mean() * 100
    print(f"Validation Accuracy @thr*: {val_acc:.2f}%")

# Test metrics principais
print(f"Final Test Accuracy: {acc*100:.2f}%")
print(f"ROC-AUC (Test): {roc_auc:.3f} | PR-AUC ({pos_label_name}): {pr_auc:.3f}")
print(f"Sensitivity ({pos_label_name}): {sens*100:.2f}% | Specificity: {spec*100:.2f}%")
print(f"PPV: {ppv*100:.2f}% | NPV: {npv*100:.2f}% | Balanced Acc: {bal_acc*100:.2f}%")
print(f"MCC: {mcc:.3f} | Brier: {brier:.4f} | Threshold*: {t_star:.3f}")
if 'local_ckpt_path' in globals():
    print(f"Model Loaded From: {local_ckpt_path}")
print("="*60)


In [ ]:
#@title 11) Plotar Matriz de Confusão
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Re-calcula a matriz de confusão usando o limiar ótimo (t_star) no conjunto de TESTE
yhat_test = (p_test >= t_star).astype(int)
cm = confusion_matrix(y_test_bin, yhat_test)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["non-mel", "mel"][:len(class_names)],
            yticklabels=["non-mel", "mel"][:len(class_names)])
plt.xlabel(f"Predição"); plt.ylabel("Real")
plt.title("Matriz de Confusão (TEST)")
plt.show()

In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path

# Caminhos para os arquivos JSON
history_file_mbnet = LOCAL_MODELS_DIR / "training_history_MobileNetV3.json"
history_file_vgg = LOCAL_MODELS_DIR / "training_history_vgg16.json" # Assuming this file exists

# Carregar dados dos arquivos JSON
history_mbnet = None
history_vgg = None

if history_file_mbnet.exists():
    try:
        with open(history_file_mbnet, "r") as f:
            history_mbnet = json.load(f)
        print(f"Dados carregados de: {history_file_mbnet}")
    except Exception as e:
        print(f"Erro ao carregar {history_file_mbnet}: {e}")
else:
    print(f"Arquivo não encontrado: {history_file_mbnet}")

if history_file_vgg.exists():
    try:
        with open(history_file_vgg, "r") as f:
            history_vgg = json.load(f)
        print(f"Dados carregados de: {history_file_vgg}")
    except Exception as e:
        print(f"Erro ao carregar {history_file_vgg}: {e}")
else:
    print(f"Arquivo não encontrado: {history_file_vgg}")

# Plotar curvas
if history_mbnet or history_vgg:
    # Plot Loss
    plt.figure(figsize=(6, 5))
    if history_mbnet and "train_losses" in history_mbnet and "val_losses" in history_mbnet:
        plt.plot(history_mbnet["train_losses"], label="MobileNetV3 Train Loss")
        plt.plot(history_mbnet["val_losses"], label="MobileNetV3 Val Loss")
    if history_vgg and "train_losses" in history_vgg and "val_losses" in history_vgg:
        plt.plot(history_vgg["train_losses"], label="VGG16 Train Loss")
        plt.plot(history_vgg["val_losses"], label="VGG16 Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training and Validation Loss")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

    # Plot Accuracy
    plt.figure(figsize=(6, 5))
    if history_mbnet and "train_accs" in history_mbnet and "val_accs" in history_mbnet:
        plt.plot(history_mbnet["train_accs"], label="MobileNetV3 Train Accuracy")
        plt.plot(history_mbnet["val_accs"], label="MobileNetV3 Val Accuracy")
    if history_vgg and "train_accs" in history_vgg and "val_accs" in history_vgg:
        plt.plot(history_vgg["train_accs"], label="VGG16 Train Accuracy")
        plt.plot(history_vgg["val_accs"], label="VGG16 Val Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.title("Training and Validation Accuracy")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
else:
    print("Nenhum arquivo de histórico válido encontrado para plotar.")

# Task
Compare the evaluation metrics of the models saved in "MobileNetV3_best_model.pth" and "vgg16_best_model.pth".

## Load mobilenetv3

### Subtask:
Load the MobileNetV3 model using its checkpoint file.


**Reasoning**:
Set the checkpoint name and force the architecture to load the MobileNetV3 model.



In [ ]:
#@title 3) Escolher o checkpoint (.pth) no Drive
from datetime import datetime

# Opção A) informe explicitamente o nome do arquivo .pth:
ckpt_name = "MobileNetV3_SkinLesion_Binary_Classifier.pth"  #@param {type:"string"}
# Ex.: "VGG16_SkinLesion_Binary_Classifier.pth" ou "MobileNetV3_SkinLesion_Binary_Classifier_1.pth"

# Opção B) se deixar vazio, tento buscar o MAIS RECENTE automaticamente:
def find_latest_pth(folder: Path):
    files = list(folder.glob("*.pth"))
    if not files:
        return None
    files.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return files[0]

ckpt_path = LOCAL_MODELS_DIR / ckpt_name if ckpt_name else find_latest_pth(DRIVE_MODELS_DIR)
assert ckpt_path and ckpt_path.exists(), "Não encontrei arquivo .pth no caminho indicado."

print("Checkpoint selecionado:", ckpt_path.name,
      f"({datetime.fromtimestamp(ckpt_path.stat().st_mtime)})")

# (opcional) copiar para pasta local por I/O mais rápido
local_ckpt_path = LOCAL_MODELS_DIR / ckpt_path.name
if not local_ckpt_path.exists():
    local_ckpt_path.write_bytes(ckpt_path.read_bytes())
print("Checkpoint local:", local_ckpt_path)

#@title 4) Detectar arquitetura, reconstruir e carregar pesos
# Detecta pelo nome do arquivo, mas você pode forçar em 'force_arch'
force_arch = "mobilenetv3-large"  #@param ["", "vgg16", "mobilenetv3-large"] {allow-input: true}
num_classes = 2  #@param {type:"integer"}

name_lower = local_ckpt_path.name.lower()
if force_arch:
    arch = force_arch
elif "vgg16" in name_lower or "vgg" in name_lower:
    arch = "vgg16"
elif "mobilenetv3" in name_lower:
    arch = "mobilenetv3-large"
else:
    # fallback seguro: peça para informar
    arch = "vgg16"  # <- troque se necessário

model, model_name = create_model(arch, num_classes=num_classes)
model = load_state_dict_safely(model, local_ckpt_path)
model.eval()

print(f"✅ Modelo carregado: {model_name} | num_classes={num_classes}")

## Calculate mobilenetv3 metrics

### Subtask:
Calculate the evaluation metrics (accuracy, ROC-AUC, etc.) for the MobileNetV3 model on the test set.


**Reasoning**:
Calculate evaluation metrics for the MobileNetV3 model on the test set as per the instructions.



In [ ]:
# 1. Get probabilities and labels for test set
p_test, y_test = probs_and_labels(model, test_loader, pos_label)

# 2. Convert true labels to binary
y_test_bin = (y_test == pos_label).astype(int)

# 3. Calculate optimal threshold using validation set (already done in previous cell)
# fpr, tpr, thr = roc_curve(y_val_bin, p_val)
# best_idx = int(np.argmax(tpr - fpr))
# t_star = float(thr[best_idx])

# 4. Generate binary predictions on test set
yhat_test = (p_test >= t_star).astype(int)

# 5. Calculate confusion matrix
cm = confusion_matrix(y_test_bin, yhat_test)

# 6. Extract TN, FP, FN, TP
tn, fp, fn, tp = cm.ravel()

# 7. Calculate metrics
epsilon = 1e-12
sens = tp / (tp + fn + epsilon)
spec = tn / (tn + fp + epsilon)
ppv = tp / (tp + fp + epsilon)
npv = tn / (tn + fn + epsilon)
acc = (yhat_test == y_test_bin).mean()
bal_acc = 0.5 * (sens + spec)

# 8. Calculate ROC-AUC
roc_auc = roc_auc_score(y_test_bin, p_test)

# 9. Calculate PR-AUC
pr_auc = average_precision_score(y_test_bin, p_test)

# 10. Calculate MCC
mcc = matthews_corrcoef(y_test_bin, yhat_test)

# 11. Calculate Cohen's Kappa
kappa = cohen_kappa_score(y_test_bin, yhat_test)

# 12. Calculate Brier score
brier = brier_score_loss(y_test_bin, p_test)

# 13. Store metrics in a dictionary
mobilenet_metrics = {
    'MobileNetV3_Accuracy': acc,
    'MobileNetV3_Balanced_Accuracy': bal_acc,
    'MobileNetV3_ROC_AUC': roc_auc,
    'MobileNetV3_PR_AUC': pr_auc,
    'MobileNetV3_Sensitivity': sens,
    'MobileNetV3_Specificity': spec,
    'MobileNetV3_PPV': ppv,
    'MobileNetV3_NPV': npv,
    'MobileNetV3_MCC': mcc,
    'MobileNetV3_Cohen_Kappa': kappa,
    'MobileNetV3_Brier_Loss': brier,
    'MobileNetV3_Optimal_Threshold': t_star
}

print("MobileNetV3 Test Metrics:")
for metric, value in mobilenet_metrics.items():
    print(f"{metric}: {value:.4f}")


## Load vgg16

### Subtask:
Load the VGG16 model using its checkpoint file.


**Reasoning**:
Set the ckpt_name and force_arch variables to load the VGG16 model from its checkpoint file.



In [ ]:
#@title 3) Escolher o checkpoint (.pth) no Drive
from datetime import datetime

# Opção A) informe explicitamente o nome do arquivo .pth:
ckpt_name = "VGG16_SkinLesion_Binary_Classifier.pth"  #@param {type:"string"}
# Ex.: "VGG16_SkinLesion_Binary_Classifier.pth" ou "MobileNetV3_SkinLesion_Binary_Classifier_1.pth"

# Opção B) se deixar vazio, tento buscar o MAIS RECENTE automaticamente:
def find_latest_pth(folder: Path):
    files = list(folder.glob("*.pth"))
    if not files:
        return None
    files.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return files[0]

ckpt_path = LOCAL_MODELS_DIR / ckpt_name if ckpt_name else find_latest_pth(DRIVE_MODELS_DIR)
assert ckpt_path and ckpt_path.exists(), "Não encontrei arquivo .pth no caminho indicado."

print("Checkpoint selecionado:", ckpt_path.name,
      f"({datetime.fromtimestamp(ckpt_path.stat().st_mtime)})")

# (opcional) copiar para pasta local por I/O mais rápido
local_ckpt_path = LOCAL_MODELS_DIR / ckpt_path.name
if not local_ckpt_path.exists():
    local_ckpt_path.write_bytes(ckpt_path.read_bytes())
print("Checkpoint local:", local_ckpt_path)

#@title 4) Detectar arquitetura, reconstruir e carregar pesos
# Detecta pelo nome do arquivo, mas você pode forçar em 'force_arch'
force_arch = "vgg16"  #@param ["", "vgg16", "mobilenetv3-large"] {allow-input: true}
num_classes = 2  #@param {type:"integer"}

name_lower = local_ckpt_path.name.lower()
if force_arch:
    arch = force_arch
elif "vgg16" in name_lower or "vgg" in name_lower:
    arch = "vgg16"
elif "mobilenetv3" in name_lower:
    arch = "mobilenetv3-large"
else:
    # fallback seguro: peça para informar
    arch = "vgg16"  # <- troque se necessário

model, model_name = create_model(arch, num_classes=num_classes)
model = load_state_dict_safely(model, local_ckpt_path)
model.eval()

print(f"✅ Modelo carregado: {model_name} | num_classes={num_classes}")

## Calculate vgg16 metrics

### Subtask:
Calculate the evaluation metrics for the VGG16 model on the test set.


**Reasoning**:
Calculate the evaluation metrics for the VGG16 model on the test set as requested in the instructions. This involves getting predictions, calculating the optimal threshold using the validation set, generating binary predictions on the test set, calculating the confusion matrix, extracting the components, and computing the various evaluation metrics.



In [ ]:
# 1. Get probabilities and labels for test set using the loaded VGG16 model
p_test_vgg, y_test_vgg = probs_and_labels(model, test_loader, pos_label)

# 2. Convert true labels to binary for VGG16 test set
y_test_bin_vgg = (y_test_vgg == pos_label).astype(int)

# 3. Get probabilities and labels for validation set using the loaded VGG16 model
p_val_vgg, y_val_vgg = probs_and_labels(model, val_loader, pos_label)

# Convert true labels to binary for VGG16 validation set
y_val_bin_vgg = (y_val_vgg == pos_label).astype(int)

# Calculate optimal threshold for VGG16 using validation set (Youden's J statistic)
fpr_vgg, tpr_vgg, thr_vgg = roc_curve(y_val_bin_vgg, p_val_vgg)
best_idx_vgg = int(np.argmax(tpr_vgg - fpr_vgg))
t_star_vgg = float(thr_vgg[best_idx_vgg])

# 4. Generate binary predictions on VGG16 test set using the optimal threshold
yhat_test_vgg = (p_test_vgg >= t_star_vgg).astype(int)

# 5. Calculate confusion matrix for VGG16
cm_vgg = confusion_matrix(y_test_bin_vgg, yhat_test_vgg)

# 6. Extract TN, FP, FN, TP for VGG16
tn_vgg, fp_vgg, fn_vgg, tp_vgg = cm_vgg.ravel()

# 7. Calculate metrics for VGG16
epsilon = 1e-12
sens_vgg = tp_vgg / (tp_vgg + fn_vgg + epsilon)
spec_vgg = tn_vgg / (tn_vgg + fp_vgg + epsilon)
ppv_vgg = tp_vgg / (tp_vgg + fp_vgg + epsilon)
npv_vgg = tn_vgg / (tn_vgg + fn_vgg + epsilon)
acc_vgg = (yhat_test_vgg == y_test_bin_vgg).mean()
bal_acc_vgg = 0.5 * (sens_vgg + spec_vgg)

# Calculate ROC-AUC for VGG16
roc_auc_vgg = roc_auc_score(y_test_bin_vgg, p_test_vgg)

# Calculate PR-AUC for VGG16
pr_auc_vgg = average_precision_score(y_test_bin_vgg, p_test_vgg)

# Calculate MCC for VGG16
mcc_vgg = matthews_corrcoef(y_test_bin_vgg, yhat_test_vgg)

# Calculate Cohen's Kappa for VGG16
kappa_vgg = cohen_kappa_score(y_test_bin_vgg, yhat_test_vgg)

# Calculate Brier score for VGG16
brier_vgg = brier_score_loss(y_test_bin_vgg, p_test_vgg)

# 8. Store metrics in a dictionary with "VGG16_" prefix
vgg16_metrics = {
    'VGG16_Accuracy': acc_vgg,
    'VGG16_Balanced_Accuracy': bal_acc_vgg,
    'VGG16_ROC_AUC': roc_auc_vgg,
    'VGG16_PR_AUC': pr_auc_vgg,
    'VGG16_Sensitivity': sens_vgg,
    'VGG16_Specificity': spec_vgg,
    'VGG16_PPV': ppv_vgg,
    'VGG16_NPV': npv_vgg,
    'VGG16_MCC': mcc_vgg,
    'VGG16_Cohen_Kappa': kappa_vgg,
    'VGG16_Brier_Loss': brier_vgg,
    'VGG16_Optimal_Threshold': t_star_vgg
}

# 9. Print the calculated VGG16 test metrics
print("VGG16 Test Metrics:")
for metric, value in vgg16_metrics.items():
    print(f"{metric}: {value:.4f}")

## Compare metrics

### Subtask:
Create a table or DataFrame to display and compare the calculated metrics for both models.


**Reasoning**:
Create a pandas DataFrame from the calculated metrics for both models, transpose it, and then display it.



In [ ]:
import pandas as pd

# Combine the metrics into a single dictionary
all_metrics = {**mobilenet_metrics, **vgg16_metrics}

# Create a DataFrame from the combined metrics
metrics_df = pd.DataFrame.from_dict(all_metrics, orient='index', columns=['Value'])

# Transpose the DataFrame
metrics_df_transposed = metrics_df.T

# Print the transposed DataFrame
display(metrics_df_transposed)

## Summary:

### Data Analysis Key Findings

*   The MobileNetV3 model achieved an Accuracy of 0.8682, a Balanced Accuracy of 0.8681, and an ROC AUC of 0.9392 on the test set.
*   The VGG16 model achieved an Accuracy of 0.9021, a Balanced Accuracy of 0.9022, and an ROC AUC of 0.9615 on the test set.
*   VGG16 showed higher Sensitivity (0.9100 vs 0.8700) and Specificity (0.8943 vs 0.8663) compared to MobileNetV3.
*   VGG16 also outperformed MobileNetV3 in terms of Precision (PPV) (0.8949 vs 0.8674) and Negative Predictive Value (NPV) (0.9095 vs 0.8690).
*   The Matthews Correlation Coefficient (MCC) and Cohen's Kappa were higher for VGG16 (0.8044 and 0.8043 respectively) than for MobileNetV3 (0.7364 and 0.7363 respectively).
*   The Brier Loss was lower for VGG16 (0.0757) than for MobileNetV3 (0.0987), indicating better calibration for VGG16.
*   The optimal threshold calculated for VGG16 using the validation set was 0.4277, while for MobileNetV3 it was 0.4232.

### Insights or Next Steps

*   The VGG16 model generally shows better performance across various evaluation metrics compared to the MobileNetV3 model on this specific dataset.
*   Further investigation could involve analyzing the types of errors made by each model (e.g., false positives vs. false negatives) to understand their respective strengths and weaknesses in more detail.


## Summary:

### Data Analysis Key Findings

* The MobileNetV3 model achieved an Accuracy of {mobilenet_metrics['MobileNetV3_Accuracy']:.4f}, a Balanced Accuracy of {mobilenet_metrics['MobileNetV3_Balanced_Accuracy']:.4f}, and an ROC AUC of {mobilenet_metrics['MobileNetV3_ROC_AUC']:.4f} on the test set.
* The VGG16 model achieved an Accuracy of {vgg16_metrics['VGG16_Accuracy']:.4f}, a Balanced Accuracy of {vgg16_metrics['VGG16_Balanced_Accuracy']:.4f}, and an ROC AUC of {vgg16_metrics['VGG16_ROC_AUC']:.4f} on the test set.
* VGG16 showed higher Sensitivity ({vgg16_metrics['VGG16_Sensitivity']:.4f} vs {mobilenet_metrics['MobileNetV3_Sensitivity']:.4f}) and Specificity ({vgg16_metrics['VGG16_Specificity']:.4f} vs {mobilenet_metrics['MobileNetV3_Specificity']:.4f}) compared to MobileNetV3.
* VGG16 also outperformed MobileNetV3 in terms of Precision (PPV) ({vgg16_metrics['VGG16_PPV']:.4f} vs {mobilenet_metrics['MobileNetV3_PPV']:.4f}) and Negative Predictive Value (NPV) ({vgg16_metrics['VGG16_NPV']:.4f} vs {mobilenet_metrics['MobileNetV3_NPV']:.4f}).
* The Matthews Correlation Coefficient (MCC) and Cohen's Kappa were higher for VGG16 ({vgg16_metrics['VGG16_MCC']:.4f} and {vgg16_metrics['VGG16_Cohen_Kappa']:.4f} respectively) than for MobileNetV3 ({mobilenet_metrics['MobileNetV3_MCC']:.4f} and {mobilenet_metrics['MobileNetV3_Cohen_Kappa']:.4f} respectively).
* The Brier Loss was lower for VGG16 ({vgg16_metrics['VGG16_Brier_Loss']:.4f}) than for MobileNetV3 ({mobilenet_metrics['MobileNetV3_Brier_Loss']:.4f}), indicating better calibration for VGG16.
* The optimal threshold calculated for VGG16 using the validation set was {vgg16_metrics['VGG16_Optimal_Threshold']:.4f}, while for MobileNetV3 it was {mobilenet_metrics['MobileNetV3_Optimal_Threshold']:.4f}.

### Metrics Comparison

{metrics_df_transposed.to_markdown(index=False)}

### Insights or Next Steps

* The VGG16 model generally shows better performance across various evaluation metrics compared to the MobileNetV3 model on this specific dataset.
* Further investigation could involve analyzing the types of errors made by each model (e.g., false positives vs. false negatives) to understand their respective strengths and weaknesses in more detail.